# Explainable PTCG Agent with Legal Ogerpon Deck

This notebook presents a reproducible baseline for **The Pokémon Company - PTCG AI Battle Challenge Simulation**.

The goal is not to start directly from a black-box reinforcement learning policy. Instead, this notebook focuses on three engineering foundations that are necessary before more advanced methods become useful:

1. Build a legal 60-card deck from the official card pool.
2. Implement the official `agent(obs_dict)` interface correctly.
3. Use an explainable scoring-based action selector that can be debugged and improved.

The submitted package passed Kaggle validation with a score of **600.0**.

> This is intended as a clear baseline and engineering reference, not a final optimized agent.

## 1. What this notebook covers

This notebook covers the full path from a legal deck to a validated submission:

- Official submission interface
- Legal deck construction
- Common validation issues
- Explainable scoring-agent design
- Mock decision evaluation
- Submission package structure
- Practical next steps

The main lesson is simple: before reinforcement learning or MCTS can help, the agent must first satisfy the official runtime protocol and deck legality constraints.

In [ ]:
from pathlib import Path
import os
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

print("Notebook working directory:", Path.cwd())
print("Available input files:")

input_root = Path("/kaggle/input")
if input_root.exists():
    files = list(input_root.rglob("*"))
    for p in files[:50]:
        if p.is_file():
            print("-", p)
    if len(files) > 50:
        print(f"... {len(files) - 50} more paths")
else:
    print("No /kaggle/input directory found. This notebook is still self-contained.")

## 2. Official submission interface

The official sample submission uses a single entry point:

```python
def agent(obs_dict: dict) -> list[int]:
    ...
```

There are two phases.

### Initial deck selection

When `obs.select is None`, the agent must return a legal 60-card deck as a list of official card IDs.

### Action selection

During gameplay, the agent must return a **list of selected option indices**, not a single integer.

Each selected index must satisfy:

```text
0 <= index < len(obs.select.option)
```

The number of selected indices must satisfy:

```text
obs.select.minCount <= len(action) <= obs.select.maxCount
```

This is one of the easiest places to make a validation error.

In [ ]:
def official_agent_contract_summary():
    contract = {
        "entry_point": "agent(obs_dict)",
        "initial_phase": "if obs.select is None: return list[int] with 60 card IDs",
        "action_phase": "return list[int] of legal option indices",
        "index_rule": "0 <= index < len(obs.select.option)",
        "count_rule": "obs.select.minCount <= len(action) <= obs.select.maxCount",
        "no_duplicates": "selected option indices must be unique",
    }
    return pd.DataFrame([{"item": k, "requirement": v} for k, v in contract.items()])

official_agent_contract_summary()

## 3. Legal Ogerpon baseline deck

The deck is written first in a human-readable format, then converted into the official `deck.csv` format using official card IDs.

The current legal baseline keeps only one ACE SPEC card and avoids Unicode card-name corruption.

Important validation lessons:

- The deck must contain exactly 60 cards.
- Most non-basic-energy cards cannot exceed 4 copies.
- ACE SPEC cards should be limited to 1 total card.
- Official card names must match exactly.
- Curly apostrophes matter, for example:
  - `Boss’s Orders`
  - `Explorer’s Guidance`

If these names become `Boss?s Orders` or `Explorer?s Guidance`, card-ID mapping fails.

In [ ]:
deck = {
    "Teal Mask Ogerpon ex": 4,
    "Teal Mask Ogerpon": 4,
    "Fezandipiti ex": 2,
    "Fan Rotom": 2,

    "Bug Catching Set": 4,
    "Ultra Ball": 4,
    "Energy Search": 4,
    "Switch": 3,
    "Boss’s Orders": 4,
    "Explorer’s Guidance": 3,
    "Billy & O'Nare": 2,
    "Perrin": 2,
    "Night Stretcher": 2,
    "Energy Retrieval": 3,
    "Energy Recycler": 2,
    "Air Balloon": 2,
    "Prime Catcher": 1,

    "Basic {G} Energy": 12,
}

deck_df = pd.DataFrame(
    [{"card_name": card, "count": count} for card, count in deck.items()]
)

total_cards = deck_df["count"].sum()
unique_cards = len(deck_df)

print("Total cards:", total_cards)
print("Unique entries:", unique_cards)

deck_df

In [ ]:
def classify_card(card_name: str) -> str:
    pokemon_names = {
        "Teal Mask Ogerpon ex",
        "Teal Mask Ogerpon",
        "Fezandipiti ex",
        "Fan Rotom",
    }

    if card_name in pokemon_names:
        return "Pokemon"
    if "Energy" in card_name and card_name.startswith("Basic"):
        return "Energy"
    return "Trainer"

deck_df["class"] = deck_df["card_name"].map(classify_card)

class_summary = (
    deck_df.groupby("class", as_index=False)["count"]
    .sum()
    .sort_values("count", ascending=False)
)

class_summary["percentage"] = class_summary["count"] / class_summary["count"].sum()

class_summary

In [ ]:
ace_spec_cards = {"Prime Catcher", "Maximum Belt"}

def validate_deck(deck_dict: dict) -> pd.DataFrame:
    checks = []

    total = sum(deck_dict.values())
    checks.append({
        "check": "Total cards equals 60",
        "passed": total == 60,
        "detail": f"total={total}",
    })

    has_question_mark = any("?" in card for card in deck_dict)
    checks.append({
        "check": "No corrupted card names with '?'",
        "passed": not has_question_mark,
        "detail": "bad names=" + str([card for card in deck_dict if "?" in card]),
    })

    ace_count = sum(count for card, count in deck_dict.items() if card in ace_spec_cards)
    checks.append({
        "check": "ACE SPEC count <= 1",
        "passed": ace_count <= 1,
        "detail": f"ace_spec_count={ace_count}",
    })

    non_basic_over_4 = [
        (card, count)
        for card, count in deck_dict.items()
        if card != "Basic {G} Energy" and count > 4
    ]
    checks.append({
        "check": "No non-basic-energy card has more than 4 copies",
        "passed": len(non_basic_over_4) == 0,
        "detail": str(non_basic_over_4),
    })

    return pd.DataFrame(checks)

validate_deck(deck)

## 4. Explainable scoring-agent design

The agent does not select actions randomly. It scores each candidate action using transparent strategic features.

The scoring features are designed to represent practical PTCG priorities:

- taking prize cards
- enabling attacks
- increasing damage pressure
- attaching or recovering energy
- setting up the bench
- using search and draw cards
- switching when poorly positioned
- using gust effects for prize routes
- recovering resources in late-game states

This design makes the agent easier to debug because a selected action can be explained by its score components.

In [ ]:
scoring_features = pd.DataFrame(
    [
        {
            "feature": "prize_route",
            "purpose": "Prioritize actions that move toward taking prize cards",
        },
        {
            "feature": "knockout",
            "purpose": "Prefer attacks that can knock out the opponent's active Pokémon",
        },
        {
            "feature": "damage_pressure",
            "purpose": "Reward actions that increase offensive pressure",
        },
        {
            "feature": "energy_readiness",
            "purpose": "Prefer actions that enable attacks through energy attachment",
        },
        {
            "feature": "bench_setup",
            "purpose": "Reward setup actions when the board is underdeveloped",
        },
        {
            "feature": "search_draw",
            "purpose": "Reward actions that improve hand quality and consistency",
        },
        {
            "feature": "switching",
            "purpose": "Reward switching when the active Pokémon is poorly positioned",
        },
        {
            "feature": "recovery",
            "purpose": "Reward late-game resource recovery",
        },
    ]
)

scoring_features

In [ ]:
from dataclasses import dataclass, field

@dataclass
class GameSnapshot:
    turn: int
    active_energy: int
    required_attack_energy: int
    opponent_active_hp: int
    bench_count: int
    discard_energy_count: int
    prizes_taken: int

@dataclass
class CandidateAction:
    action_id: str
    action_type: str
    card_name: str = ""
    damage: int = 0
    energy_gain: int = 0
    draw_value: int = 0
    search_value: int = 0
    switch_value: int = 0
    recovery_value: int = 0
    gust_value: int = 0
    tags: set = field(default_factory=set)

def score_action(state: GameSnapshot, action: CandidateAction) -> tuple[float, list[str]]:
    score = 0.0
    reasons = []

    if action.action_type == "attack":
        score += 0.20 * action.damage
        reasons.append(f"damage pressure +{0.20 * action.damage:.1f}")

        if action.damage >= state.opponent_active_hp:
            score += 100
            reasons.append("knockout opportunity +100")

    if action.energy_gain > 0:
        score += 18 * action.energy_gain
        reasons.append(f"energy gain +{18 * action.energy_gain}")

        if state.active_energy + action.energy_gain >= state.required_attack_energy:
            score += 25
            reasons.append("enables attack +25")

    if action.search_value > 0:
        score += 8 * action.search_value
        reasons.append(f"search consistency +{8 * action.search_value}")

    if action.draw_value > 0:
        score += 6 * action.draw_value
        reasons.append(f"draw value +{6 * action.draw_value}")

    if action.switch_value > 0:
        score += 12 * action.switch_value
        reasons.append(f"switching value +{12 * action.switch_value}")

    if action.gust_value > 0:
        score += 20 * action.gust_value
        reasons.append(f"gust route +{20 * action.gust_value}")

    if "prize_route" in action.tags:
        score += 35
        reasons.append("explicit prize route +35")

    if action.recovery_value > 0 and state.turn >= 6 and state.discard_energy_count >= 3:
        score += 18 * action.recovery_value
        reasons.append(f"late recovery +{18 * action.recovery_value}")

    if action.action_type == "end_turn":
        score -= 10
        reasons.append("avoid passive end turn -10")

    return score, reasons

In [ ]:
state = GameSnapshot(
    turn=6,
    active_energy=1,
    required_attack_energy=2,
    opponent_active_hp=120,
    bench_count=2,
    discard_energy_count=4,
    prizes_taken=3,
)

candidate_actions = [
    CandidateAction(
        action_id="attach_energy_to_ogerpon",
        action_type="attach_energy",
        card_name="Basic {G} Energy",
        energy_gain=1,
        tags={"energy", "attack_enable"},
    ),
    CandidateAction(
        action_id="boss_orders_prize_route",
        action_type="trainer",
        card_name="Boss’s Orders",
        gust_value=1,
        tags={"gust", "prize_route"},
    ),
    CandidateAction(
        action_id="energy_retrieval",
        action_type="trainer",
        card_name="Energy Retrieval",
        recovery_value=1,
        tags={"resource_loop"},
    ),
    CandidateAction(
        action_id="attack_pressure",
        action_type="attack",
        card_name="Teal Mask Ogerpon ex",
        damage=80,
    ),
    CandidateAction(
        action_id="end_turn",
        action_type="end_turn",
    ),
]

rows = []
for action in candidate_actions:
    score, reasons = score_action(state, action)
    rows.append(
        {
            "action_id": action.action_id,
            "card_name": action.card_name,
            "score": score,
            "reasons": "; ".join(reasons),
        }
    )

ranking_df = pd.DataFrame(rows).sort_values("score", ascending=False)
ranking_df

## 5. Mock decision evaluation

Before submitting to Kaggle, I used mock decision scenarios to check whether the agent selected strategically reasonable actions.

The aim was not to prove final playing strength, but to test whether the scoring logic behaves correctly in important tactical situations.

Examples of tested scenarios:

- energy development
- direct knockout
- bench setup
- gust-based prize route
- late-game resource recovery

In [ ]:
mock_results = pd.DataFrame(
    [
        {
            "scenario": "S1_energy_development",
            "expected_action": "attach_energy",
            "rule_agent": "attach_energy",
            "explainable_agent": "attach_energy",
        },
        {
            "scenario": "S2_take_knockout",
            "expected_action": "attack_knockout",
            "rule_agent": "attack_knockout",
            "explainable_agent": "attack_knockout",
        },
        {
            "scenario": "S3_setup_bench",
            "expected_action": "setup_bench",
            "rule_agent": "setup_bench",
            "explainable_agent": "setup_bench",
        },
        {
            "scenario": "S4_gust_route",
            "expected_action": "boss_orders",
            "rule_agent": "draw",
            "explainable_agent": "boss_orders",
        },
        {
            "scenario": "S5_draw_resources",
            "expected_action": "draw",
            "rule_agent": "draw",
            "explainable_agent": "draw",
        },
    ]
)

mock_results["rule_pass"] = mock_results["rule_agent"] == mock_results["expected_action"]
mock_results["explainable_pass"] = mock_results["explainable_agent"] == mock_results["expected_action"]

mock_results

In [ ]:
real_deck_results = pd.DataFrame(
    [
        {
            "scenario": "R1_ogerpon_energy_enable",
            "expected_action": "attach_energy",
            "rule_agent": "attach_energy",
            "explainable_agent": "attach_energy",
        },
        {
            "scenario": "R2_ogerpon_direct_knockout",
            "expected_action": "attack_knockout",
            "rule_agent": "attack_knockout",
            "explainable_agent": "attack_knockout",
        },
        {
            "scenario": "R3_ogerpon_setup_bench",
            "expected_action": "setup_bench",
            "rule_agent": "setup_bench",
            "explainable_agent": "setup_bench",
        },
        {
            "scenario": "R4_ogerpon_gust_prize_route",
            "expected_action": "boss_orders",
            "rule_agent": "draw",
            "explainable_agent": "boss_orders",
        },
        {
            "scenario": "R5_ogerpon_late_recovery",
            "expected_action": "recovery",
            "rule_agent": "draw",
            "explainable_agent": "recovery",
        },
    ]
)

real_deck_results["rule_pass"] = real_deck_results["rule_agent"] == real_deck_results["expected_action"]
real_deck_results["explainable_pass"] = real_deck_results["explainable_agent"] == real_deck_results["expected_action"]

summary = pd.DataFrame(
    [
        {
            "test_set": "generic_mock",
            "rule_agent_pass": int(mock_results["rule_pass"].sum()),
            "explainable_agent_pass": int(mock_results["explainable_pass"].sum()),
            "total": len(mock_results),
        },
        {
            "test_set": "real_deck_mock",
            "rule_agent_pass": int(real_deck_results["rule_pass"].sum()),
            "explainable_agent_pass": int(real_deck_results["explainable_pass"].sum()),
            "total": len(real_deck_results),
        },
    ]
)

summary

## 6. Submission package structure

The validated submission package follows the official style:

```text
submission.tar.gz
├── main.py
├── deck.csv
├── cg/
├── src/
└── configs/
```

Key implementation details:

- `main.py` exposes `agent(obs_dict)`.
- `deck.csv` contains exactly 60 official card IDs.
- The official `cg/` directory is included.
- The internal explainable agent is called through a Kaggle adapter.
- During gameplay, the agent returns `list[int]`, not `int`.

The first successful Kaggle validation produced a score of **600.0**.

In [ ]:
submission_checklist = pd.DataFrame(
    [
        {"item": "main.py defines agent(obs_dict)", "status": "passed"},
        {"item": "Initial phase returns 60-card deck list", "status": "passed"},
        {"item": "Action phase returns list[int]", "status": "passed"},
        {"item": "deck.csv has 60 official card IDs", "status": "passed"},
        {"item": "Only one ACE SPEC card is used", "status": "passed"},
        {"item": "Official cg/ directory included", "status": "passed"},
        {"item": "Kaggle validation completed", "status": "passed"},
        {"item": "Validation score", "status": "600.0"},
    ]
)

submission_checklist

## 7. Reproducibility workflow

The local engineering workflow used to build and check the submission package was:

```bash
python scripts/profile_deck_v2.py
python scripts/build_official_deck_csv.py
python scripts/build_official_style_submission.py
python scripts/check_official_style_submission.py
python scripts/run_agent_regression_suite.py
```

The final upload file was:

```text
submissions/submission.tar.gz
```

This workflow helped catch several validation issues before resubmission.

## 8. What went wrong before validation passed

The failed validation attempt was useful because it isolated the issue to deck legality.

Main issues fixed:

1. **Extra ACE SPEC cards**
   - The early deck contained more than one ACE SPEC card.
   - The corrected deck keeps only `Prime Catcher`.

2. **Unicode card-name corruption**
   - `Boss’s Orders` and `Explorer’s Guidance` were accidentally written with `?` instead of the official curly apostrophe.
   - This caused card-ID mapping failure.
   - The final deck uses the correct Unicode apostrophe.

3. **Official return format**
   - The official runtime expects `list[int]`.
   - Returning a single integer is not valid.

These are small details, but they are enough to make a validation episode fail.

## 9. Next steps

This notebook is a baseline. The next improvements should focus on playing strength rather than package validation.

Planned improvements:

- Parse official observations more accurately.
- Add replay-based failure analysis.
- Improve prize-route planning.
- Add matchup-specific scoring weights.
- Add better target selection.
- Extend from scoring rules to search-enhanced decision making.
- Use MCTS or hybrid RL only after the rule/scoring baseline is stable.

The main takeaway is that a reliable PTCG agent should be built in layers:

```text
legal deck → official interface → explainable scoring → validation → replay analysis → stronger planning
```

## 10. Conclusion

This notebook provides a validated, explainable baseline for the PTCG AI Battle Challenge Simulation track.

The contribution is not a final high-ranking policy. The contribution is a stable engineering foundation:

- legal Ogerpon-oriented deck
- official `agent(obs_dict)` interface
- transparent scoring-based action selection
- local regression checks
- successful Kaggle validation

I hope this helps other participants avoid early validation errors and build stronger agents on top of a reproducible baseline.